# Module Principles of Machine Learning 7WCM2032

### Copyright Notice

© 2024 Manal Helal, University of Hertfordshire

This Jupyter notebook is provided by Manal Helal, a lecturer at the University of Hertfordshire, for educational purposes. You are free to use, share, and modify the contents of this notebook under the following conditions:

1. **Attribution**: You must give appropriate credit, provide a link to the original source, and indicate if changes were made. You may do so in any reasonable manner, but not in any way that suggests the lecturer or the University of Hertfordshire endorses you or your use.

   **Suggested Attribution**:

This notebook was originally created by Manal Helal, University of Hertfordshire for the Principles of Machine Learning Module (7WCM2032) 2024.

2. **No Warranty**: The content of this notebook is provided "as-is," without warranty of any kind. The lecturer and the University of Hertfordshire make no representations or warranties, either express or implied, as to the accuracy, reliability, or completeness of the information provided herein.

3. **Limited Liability**: In no event shall the lecturer or the University of Hertfordshire be liable for any damages arising from the use of, or inability to use, the contents of this notebook, including but not limited to damages for loss of data, loss of profits, or interruption of business, even if advised of the possibility of such damages.

4. **Environment and Compatibility**: This notebook has been developed and tested in a specific environment. The lecturer and the University of Hertfordshire cannot guarantee that the notebook will function as expected in different environments. Users are responsible for ensuring compatibility and for addressing any issues that may arise.

For any questions or further information, please contact Manal Helal at m.helal@herts.ac.uk


## Building on Pretrained Language Models - RAG

This code accompanies the teaching content on:

- Embeddings and vector search
- Retrieval-Augmented Generation (RAG)
- Prompting strategies (zero-shot, few-shot, chain-of-thought)
- Evaluation of LLM/RAG systems
- Comparing prompting vs. retrieval vs. fine-tuning

Dependencies:

pip install sentence-transformers chromadb transformers torch

### SECTION 1: EMBEDDINGS AND VECTOR SEARCH


In [ ]:
from dataclasses import dataclass

import numpy as np


class EmbeddingModel:
    """Wrapper for generating text embeddings using sentence-transformers."""

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """Initialize embedding model.

        Args:
            model_name: Name of the sentence-transformers model
        """
        from sentence_transformers import SentenceTransformer

        self.model = SentenceTransformer(model_name)
        self.dimension = self.model.get_sentence_embedding_dimension()

    def embed(self, texts: list[str]) -> np.ndarray:
        """Generate embeddings for a list of texts.

        Args:
            texts: List of text strings to embed

        Returns:
            numpy array of shape (len(texts), dimension)
        """
        return self.model.encode(texts, normalize_embeddings=True)

    def embed_single(self, text: str) -> np.ndarray:
        """Generate embedding for a single text."""
        return self.embed([text])[0]


class SimpleVectorDatabase:
    """In-memory vector database for storing and retrieving embeddings."""

    def __init__(self, embedding_model: EmbeddingModel):
        self.embedding_model = embedding_model
        self.documents: list[str] = []
        self.metadata: list[dict] = []
        self.embeddings: np.ndarray | None = None

    def add_documents(
        self, documents: list[str], metadata: list[dict] | None = None
    ) -> None:
        """Add documents to the vector database.

        Args:
            documents: List of text documents
            metadata: Optional metadata for each document
        """
        if metadata is None:
            metadata = [{"id": i} for i in range(len(documents))]

        # Generate embeddings for all documents at once
        embeddings = self.embedding_model.embed(documents)

        self.documents.extend(documents)
        self.metadata.extend(metadata)

        if self.embeddings is None:
            self.embeddings = embeddings
        else:
            self.embeddings = np.vstack([self.embeddings, embeddings])

    def search(self, query: str, top_k: int = 4) -> list[tuple[str, float, dict]]:
        """Search for the most similar documents to a query.

        Args:
            query: Search query text
            top_k: Number of results to return

        Returns:
            List of tuples (document_text, similarity_score, metadata)
        """
        query_vec = self.embedding_model.embed_single(query)

        # Compute cosine similarity
        similarities = np.dot(self.embeddings, query_vec)

        # Get top_k indices
        top_indices = np.argsort(similarities)[-top_k:][::-1]

        results = []
        for idx in top_indices:
            results.append(
                (self.documents[idx], float(similarities[idx]), self.metadata[idx])
            )

        return results


### SECTION 2: PROMPTING STRATEGIES


In [ ]:
class PromptingStrategies:
    """Demonstrates different prompting approaches."""

    @staticmethod
    def zero_shot_prompt(question: str) -> str:
        """Zero-shot: ask directly without examples."""
        return f"Question: {question}\nAnswer:"

    @staticmethod
    def few_shot_prompt(question: str, examples: list[tuple[str, str]]) -> str:
        """Few-shot: provide examples before the question."""
        prompt = "Here are some examples:\n\n"
        for ex_q, ex_a in examples:
            prompt += f"Question: {ex_q}\nAnswer: {ex_a}\n\n"
        prompt += f"Now answer this:\nQuestion: {question}\nAnswer:"
        return prompt

    @staticmethod
    def chain_of_thought_prompt(question: str) -> str:
        """Chain-of-thought: ask for step-by-step reasoning."""
        return f"Question: {question}\n\nLet's solve this step by step:\n1."


### SECTION 3: RETRIEVAL-AUGMENTED GENERATION (RAG)


In [ ]:
@dataclass
class RAGResponse:
    """Container for RAG response with sources."""

    answer: str
    sources: list[dict]
    context_used: list[str]
    confidence: float


class SimpleRAGSystem:
    """
    A complete RAG system that:
    1. Indexes documents
    2. Retrieves relevant context
    3. Generates grounded answers
    """

    def __init__(self, vector_db: SimpleVectorDatabase, llm_model=None):
        self.vector_db = vector_db
        self.llm_model = llm_model

    def _format_context(self, retrieved_docs: list[tuple[str, float, dict]]) -> str:
        """Format retrieved documents for inclusion in the prompt."""
        formatted = []
        for i, (doc, score, meta) in enumerate(retrieved_docs, 1):
            formatted.append(f"[{i}] {doc}")
        return "\n\n".join(formatted)

    def _create_prompt(
        self, question: str, context: list[tuple[str, float, dict]]
    ) -> str:
        """Create the complete prompt with context."""
        context_str = self._format_context(context)

        prompt = """Answer using ONLY the context below. If the answer is not present, say "I don't know" or "I cannot answer this based on the provided information."

Context:
{context}

Question: {question}

Answer:"""

        return prompt.format(context=context_str, question=question)

    def answer(self, question: str, top_k: int = 4) -> RAGResponse:
        """Generate a RAG answer for a question."""
        # Retrieve relevant documents
        retrieved = self.vector_db.search(question, top_k=top_k)

        # Create the prompt
        prompt = self._create_prompt(question, retrieved)

        # Generate answer using the LLM
        if self.llm_model is not None:
            answer = self.llm_model.generate(prompt)
        else:
            # Extract the most relevant document content
            if retrieved and retrieved[0][1] > 0.2:  # Similarity threshold
                # Get the best matching document
                best_doc = retrieved[0][0]
                # Extract relevant sentences (simple heuristic)
                sentences = best_doc.split(".")
                # Find sentences that might answer the question
                relevant_sentences = []
                question_words = set(question.lower().split())
                for sent in sentences:
                    sent_words = set(sent.lower().split())
                    # Check if question words appear in the sentence
                    if len(question_words.intersection(sent_words)) > 1:
                        relevant_sentences.append(sent.strip())

                if relevant_sentences:
                    answer = ". ".join(relevant_sentences[:2]) + "."
                else:
                    # Fallback: use first 2 sentences from best document
                    answer = ". ".join(sentences[:2]) + "."
            else:
                answer = "I don't know. The answer to this question is not available in the provided documents."

        # Extract sources for transparency
        sources = [
            {
                "text": doc[:100] + "..." if len(doc) > 100 else doc,
                "score": score,
                "metadata": meta,
            }
            for doc, score, meta in retrieved
        ]

        return RAGResponse(
            answer=answer,
            sources=sources,
            context_used=[doc for doc, _, _ in retrieved],
            confidence=0.85 if retrieved and retrieved[0][1] > 0.2 else 0.1,
        )

### SECTION 4: EVALUATION FRAMEWORK


In [ ]:
class RAGEvaluator:
    """Evaluate a RAG system against test questions."""

    def __init__(self, rag_system: SimpleRAGSystem):
        self.rag_system = rag_system

    def evaluate(self, test_cases: list[dict]) -> dict:
        """Evaluate RAG system on test cases.

        Args:
            test_cases: List of dictionaries with keys:
                       - question: str
                       - expected_answer: str (optional)
                       - should_be_found: bool (is the answer in the document set?)
                       - context: str (the expected source text)

        Returns:
            Evaluation metrics
        """
        results = {
            "total": len(test_cases),
            "correct": 0,
            "hallucinations": 0,
            "citations_present": 0,
            "details": [],
        }

        for case in test_cases:
            response = self.rag_system.answer(case["question"])

            # Check if the answer is supported by context
            is_supported = self._check_support(response, case.get("context", ""))

            # Check if the system correctly declined when answer not present
            if not case.get("should_be_found", True):
                is_correct = self._check_declined(response.answer)
            else:
                is_correct = is_supported

            # Check for hallucinations (unsupported claims)
            has_hallucination = not is_supported and case.get("should_be_found", True)

            # Check if citations are present
            has_citations = len(response.sources) > 0

            if is_correct:
                results["correct"] += 1
            if has_hallucination:
                results["hallucinations"] += 1
            if has_citations:
                results["citations_present"] += 1

            results["details"].append(
                {
                    "question": case["question"],
                    "answer": response.answer,
                    "is_correct": is_correct,
                    "has_hallucination": has_hallucination,
                    "has_citations": has_citations,
                    "sources": response.sources,
                }
            )

        # Compute metrics
        results["accuracy"] = results["correct"] / results["total"]
        results["citation_rate"] = results["citations_present"] / results["total"]

        return results

    def _check_support(self, response: RAGResponse, expected_context: str) -> bool:
        """Check if the answer is supported by the retrieved context."""
        if not expected_context:
            return True  # No expected context to check

        # Simplified check: does the answer mention content from context?
        # In practice, use more sophisticated methods
        return any(
            expected_context.lower() in doc.lower() for doc in response.context_used
        )

    def _check_declined(self, answer: str) -> bool:
        """Check if the system correctly declined to answer."""
        decline_phrases = [
            "i don't know",
            "cannot answer",
            "not provided",
            "not available",
            "i cannot",
        ]
        return any(phrase in answer.lower() for phrase in decline_phrases)


### SECTION 5: COMPARING APPROACHES


In [ ]:
class ApproachComparison:
    """Compare different approaches: prompting, retrieval, fine-tuning."""

    def __init__(self, rag_system: SimpleRAGSystem):
        self.rag_system = rag_system

    def compare_approaches(
        self, question: str, few_shot_examples: list[tuple[str, str]] | None = None
    ):
        """Demonstrate different approaches on the same question."""

        approaches = {}

        # 1. Zero-shot prompting (direct)
        approaches["zero_shot"] = {
            "description": "Direct question, no context",
            "prompt": PromptingStrategies.zero_shot_prompt(question),
            "response": None,  # Would call LLM here
        }

        # 2. Few-shot prompting
        if few_shot_examples:
            approaches["few_shot"] = {
                "description": "With examples",
                "prompt": PromptingStrategies.few_shot_prompt(
                    question, few_shot_examples
                ),
                "response": None,
            }

        # 3. Chain-of-thought
        approaches["chain_of_thought"] = {
            "description": "Step-by-step reasoning",
            "prompt": PromptingStrategies.chain_of_thought_prompt(question),
            "response": None,
        }

        # 4. RAG (Retrieval-Augmented Generation)
        rag_response = self.rag_system.answer(question)
        approaches["rag"] = {
            "description": "Retrieval-augmented with context",
            "prompt": "",  # RAG doesn't use a single prompt
            "response": rag_response.answer,
            "sources": rag_response.sources,
        }

        return approaches


### SECTION 6: PRACTICAL ACTIVITY - BUILD A RAG SYSTEM


In [ ]:
class PracticalRAGActivity:
    """Step-by-step practical activity for building a RAG system."""

    @staticmethod
    def create_sample_documents() -> list[dict]:
        """Create sample documents about a fictional company."""
        return [
            {
                "id": "doc1",
                "text": """Our password reset process is simple. Users can request a password 
                reset via the login page. Click 'Forgot Password' and enter your email address. 
                A reset link will be sent within 5 minutes. The link expires after 30 minutes.""",
                "metadata": {"category": "security", "topic": "password reset"},
            },
            {
                "id": "doc2",
                "text": """Two-factor authentication (2FA) is available for all accounts. 
                To enable 2FA, go to Account Settings > Security > Two-Factor Authentication. 
                You can use either authenticator apps (like Google Authenticator) or SMS verification.""",
                "metadata": {"category": "security", "topic": "authentication"},
            },
            {
                "id": "doc3",
                "text": """API access requires a valid API key. To generate an API key: 
                1. Log in to your dashboard
                2. Navigate to Settings > API
                3. Click 'Generate New Key'
                4. Copy and save the key immediately (you won't see it again)""",
                "metadata": {"category": "api", "topic": "keys"},
            },
            {
                "id": "doc4",
                "text": """Our support hours are Monday to Friday, 9:00 AM to 6:00 PM EST. 
                For urgent issues outside these hours, please email support@example.com 
                with 'URGENT' in the subject line. Response times are typically 2-4 hours 
                for urgent issues and 24-48 hours for standard queries.""",
                "metadata": {"category": "support", "topic": "hours"},
            },
            {
                "id": "doc5",
                "text": """Data export is available for all premium accounts. You can export 
                your data in CSV or JSON format. To export: go to Dashboard > Data > Export, 
                select the data range and format, then click 'Export'. You will receive an 
                email with the download link within 10 minutes. Exports are available for 
                7 days before they are automatically deleted.""",
                "metadata": {"category": "data", "topic": "export"},
            },
        ]

    @staticmethod
    def run_activity():
        """Run the complete practical activity."""
        print("=" * 60)
        print("PRACTICAL ACTIVITY: Building a RAG System")
        print("=" * 60)

        # Step 1: Initialize embedding model
        print("\n[Step 1] Initializing embedding model...")
        embed_model = EmbeddingModel()
        print(f"Embedding dimension: {embed_model.dimension}")

        # Step 2: Create vector database
        print("\n[Step 2] Creating vector database and indexing documents...")
        vector_db = SimpleVectorDatabase(embed_model)

        documents = PracticalRAGActivity.create_sample_documents()
        doc_texts = [doc["text"] for doc in documents]
        doc_metadata = [doc["metadata"] for doc in documents]
        vector_db.add_documents(doc_texts, doc_metadata)
        print(f"Indexed {len(documents)} documents")

        # Step 3: Create RAG system
        print("\n[Step 3] Setting up RAG system...")
        rag_system = SimpleRAGSystem(vector_db)
        print("RAG system ready")

        # Step 4: Test with questions that should be answered
        print("\n[Step 4] Testing with in-domain questions...")
        test_cases = [
            {
                "question": "How do I reset my password?",
                "should_be_found": True,
                "context": "password reset process",
            },
            {
                "question": "What are the support hours?",
                "should_be_found": True,
                "context": "support hours",
            },
            {
                "question": "How do I enable two-factor authentication?",
                "should_be_found": True,
                "context": "two-factor authentication",
            },
            {
                "question": "How do I generate an API key?",
                "should_be_found": True,
                "context": "API key",
            },
            {
                "question": "How do I export my data?",
                "should_be_found": True,
                "context": "data export",
            },
        ]

        for case in test_cases:
            response = rag_system.answer(case["question"])
            print(f"\nQuestion: {case['question']}")
            print(f"Answer: {response.answer}")
            print(f"Sources: {len(response.sources)} documents retrieved")

        # Step 5: Test with out-of-domain questions (should decline)
        print("\n[Step 5] Testing with out-of-domain questions...")
        out_of_domain = [
            {
                "question": "What is the weather forecast for tomorrow?",
                "should_be_found": False,
            },
            {"question": "How do I file a tax return?", "should_be_found": False},
        ]

        for case in out_of_domain:
            response = rag_system.answer(case["question"])
            print(f"\nQuestion: {case['question']}")
            print(f"Answer: {response.answer}")
            evaluator = RAGEvaluator(rag_system)
            print(f"Declined correctly: {evaluator._check_declined(response.answer)}")

        # Step 6: Evaluate
        print("\n[Step 6] Evaluation...")
        evaluator = RAGEvaluator(rag_system)
        all_test_cases = test_cases + [
            {
                "question": "What is the weather forecast for tomorrow?",
                "should_be_found": False,
            },
            {"question": "How do I file a tax return?", "should_be_found": False},
        ]
        results = evaluator.evaluate(all_test_cases)

        print("\nEvaluation Results:")
        print(f"Total questions: {results['total']}")
        print(f"Correct: {results['correct']} ({results['accuracy'] * 100:.1f}%)")
        print(f"Hallucinations detected: {results['hallucinations']}")
        print(f"Citations provided: {results['citation_rate'] * 100:.1f}%")

        return rag_system, results


### Run the complete demonstration.


In [ ]:
# Run the practical activity
rag_system, results = PracticalRAGActivity.run_activity()

# Show comparison of approaches
print("\n[Comparing Approaches...]")
comparison = ApproachComparison(rag_system)
approaches = comparison.compare_approaches(
    "How do I reset my password?",
    few_shot_examples=[
        (
            "How do I enable 2FA?",
            "Go to Account Settings > Security > Two-Factor Authentication",
        ),
        ("How do I generate an API key?", "Go to Settings > API > Generate New Key"),
    ],
)

print("\nApproaches compared:")
for name, details in approaches.items():
    print(f"\n{name.upper().replace('_', ' ')}:")
    print(f"  Description: {details['description']}")
    if details.get("response"):
        print(f"  Response: {details['response']}")


PRACTICAL ACTIVITY: Building a RAG System

[Step 1] Initializing embedding model...
Embedding dimension: 384

[Step 2] Creating vector database and indexing documents...
Indexed 5 documents

[Step 3] Setting up RAG system...
RAG system ready

[Step 4] Testing with in-domain questions...

Question: How do I reset my password?
Answer: Our password reset process is simple.  Users can request a password 
                reset via the login page.
Sources: 4 documents retrieved

Question: What are the support hours?
Answer: Our support hours are Monday to Friday, 9:00 AM to 6:00 PM EST.
Sources: 4 documents retrieved

Question: How do I enable two-factor authentication?
Answer: To enable 2FA, go to Account Settings > Security > Two-Factor Authentication.
Sources: 4 documents retrieved

Question: How do I generate an API key?
Answer: To generate an API key: 
                1.
Sources: 4 documents retrieved

Question: How do I export my data?
Answer: Data export is available for all premium a

# Exercise: ADD an LLM model

Set up API credentials for NVIDIA NIM and OpenAI to run LLM inference in Python. You’ll learn how to securely manage keys, connect to both services, and verify that everything works—skills essential for building real-world applications with pretrained models.

## Prerequisites

- Python 3.10+ installed
- An account with OpenAI (with credits) and NVIDIA (free tier available)
- Your existing code environment from the RAG exercise

## OpenAI API Setup

### Get Your OpenAI API Key

- Go to <a href="https://platform.openai.com/login?next=%2Fapi-keys">platform.openai.com/api-keys</a>
- Click Create new secret key
- Give it a name (e.g., "RAG-Exercise") and select All permissions
- Copy the key immediately—you won’t see it again!

### Set the Environment Variable

On macOS/Linux (bash/zsh):

export OPENAI_API_KEY="sk-proj-..." # Replace with your actual key

On Windows (PowerShell):

env:OPENAI_API_KEY = "sk-proj-..."

To make it persist across sessions, add the export line to your ~/.bashrc or ~/.zshrc.

### Install and Verify

pip install openai

### Create a file test_openai.py:

<pre>
from openai import OpenAI
import os

client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

response = client.chat.completions.create(
    model="gpt-3.5-turbo",
    messages=[{"role": "user", "content": "Say 'Hello, API is working!'"}]
)
print(response.choices[0].message.content)

</pre>

Run it: python test_openai.py → Expected output: Hello, API is working!

## NVIDIA NIM Setup

### Get Your NVIDIA NIM API Key

- Go to <a href="https://build.nvidia.com/">build.nvidia.com</a> and sign in
- Navigate to any model page (e.g., meta/llama-3.1-70b-instruct)
- Click Get API Key
- Copy your key (it starts with nvapi-)

### Set the Environment Variable

On macOS/Linux:

export NVIDIA_API_KEY="nvapi-..." # Or NVIDIA_NIM_API_KEY

On Windows (PowerShell):

env:NVIDIA_API_KEY = "nvapi-..."

### Connect and Test

NVIDIA NIM provides an OpenAI-compatible API, so you can use the same openai library!

<pre>
from openai import OpenAI
import os

client = OpenAI(
    base_url="https://integrate.api.nvidia.com/v1",
    api_key=os.environ.get("NVIDIA_API_KEY")
)

response = client.chat.completions.create(
    model="meta/llama-3.1-70b-instruct",
    messages=[{"role": "user", "content": "Say 'Hello from NVIDIA NIM!'"}],
    temperature=0.7,
    max_tokens=256
)
print(response.choices[0].message.content)

</pre>

### Key differences from OpenAI:

- Base URL: https://integrate.api.nvidia.com/v1 (do not click, use in the code below)
- Model names: e.g., meta/llama-3.1-70b-instruct (check the model page for exact name)
- Free tier: ~40 requests/minute per model

### Compare Both Services

Create a script that sends the same prompt to both services and compares the responses:

<pre>

from openai import OpenAI
import os

def query_openai(prompt):
    client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))
    response = client.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content

def query_nvidia(prompt):
    client = OpenAI(
        base_url="https://integrate.api.nvidia.com/v1",
        api_key=os.environ.get("NVIDIA_API_KEY")
    )
    response = client.chat.completions.create(
        model="meta/llama-3.1-70b-instruct",
        messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content

prompt = "Explain Retrieval-Augmented Generation in one paragraph."
print("OpenAI:", query_openai(prompt))
print("\nNVIDIA:", query_nvidia(prompt))

</pre>

### Integrate with Your RAG System

Finally, replace your mock LLM in the RAG system with real API calls:

<pre>
class RealLLM:
    def __init__(self, provider="openai"):
        self.provider = provider
        if provider == "openai":
            self.client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))
            self.model = "gpt-3.5-turbo"
        else:  # nvidia
            self.client = OpenAI(
                base_url="https://integrate.api.nvidia.com/v1",
                api_key=os.environ.get("NVIDIA_API_KEY")
            )
            self.model = "meta/llama-3.1-70b-instruct"
    
    def generate(self, prompt):
        response = self.client.chat.completions.create(
            model=self.model,
            messages=[{"role": "user", "content": prompt}]
        )
        return response.choices[0].message.content


</pre>

### Use in your RAG system:

<pre>
llm = RealLLM(provider="nvidia")  # or "openai"
rag_system = SimpleRAGSystem(vector_db, llm_model=llm)
response = rag_system.answer("How do I reset my password?")
print(response.answer)


</pre>

### Success Checklist

- OpenAI API key obtained and stored as OPENAI_API_KEY
- NVIDIA NIM API key obtained and stored as NVIDIA_API_KEY
- pip install openai completed
- OpenAI test script returns a valid response
- NVIDIA NIM test script returns a valid response
- Both services integrated into your RAG system
- Comparison script run successfully


# Exercise: Add Large documents

In your coursework, you can add an encyclopedia pdf and ask questions in its context. For example:

- https://www.wikidata.org/wiki/File:A_dictionary_of_medicine_(electronic_resource)_-_including_general_pathology,_general_therapeutics,_hygiene,_and_the_diseases_peculiar_to_women_and_children_(IA_b20420699_002).pdf

- https://commons.wikimedia.org/wiki/File:The_family_physician,_or,_Every_man_his_own_doctor_(microform)_-_an_encyclopedia_of_medicine,_containing_knowledge_that_will_promote_health,_cure_disease_and_prolong_life,_describing_all_diseases..._(IA_cihm_02946).pdf?uselang=fr

- https://www.wikidata.org/wiki/File:The_mechanical_engineer%27s_reference_book;_a_handbook_of_tables,_formulas,_and_methods_for_engineers,_students,_and_draftsmen_(IA_mechanicalengine00supl).pdf

- https://opus.lib.uts.edu.au/handle/10453/23795

- https://liverpooluniversitypress.co.uk/essentials-of-financial-management-efm.html
